## 전역

In [1]:
# ============================================================
# Top10 1위 조합 기반 SHAP + Permutation Importance
#   -> feature_analysis_{FEATURE_SET}.csv + PNG 3개만 저장
# ============================================================

import os
import platform
import warnings
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. Top10_우수모델.csv 1번째 행 정보 로드
# ============================================================
TOP10_PATH = "15번. 우수모델 데이터/Top10 우수모델.csv"

top10 = pd.read_csv(TOP10_PATH, index_col=0)
row = top10.iloc[0]

FEATURE_SET  = row["FeatureSet"]
FEATURE_FILE = row["FeatureFile"]
METHOD       = row["Method"]
SMOTE_RATIO  = row["SMOTE_Ratio"]

print(f"FeatureSet: {FEATURE_SET} | Method: {METHOD} | SMOTE_Ratio: {SMOTE_RATIO}")


# ============================================================
# 2. 경로 / 설정값
# ============================================================
TRAIN_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = os.path.join(r'13번.피처셀렉션\M19_도매_소매업', FEATURE_FILE)

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42

SHAP_SAVE_DIR = r"16번. SHAP\전역"
save_sub = os.path.join(SHAP_SAVE_DIR, FEATURE_SET)
os.makedirs(save_sub, exist_ok=True)


# ============================================================
# 3. Method / SMOTE_Ratio 기반 불균형 처리 함수
# ============================================================
def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    minority   = X[y == 1].copy()
    n_minority = len(minority)
    n_majority = (y == 0).sum()
    target_n = int(n_majority * ratio) if ratio else n_majority
    n_synth  = max(0, target_n - n_minority)
    if n_synth == 0 or n_minority < 5:
        return X, y
    ctgan = CTGAN(epochs=300, verbose=False)
    ctgan.fit(minority)
    synth = ctgan.sample(n_synth)
    synth = synth[X.columns]
    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("BorderlineSMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("SMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


# ============================================================
# 4. 데이터 로드 + 모델 학습
# ============================================================
train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

print(f"피처 수: {len(use_features)}개")

imputer = SimpleImputer(strategy="median")
X_train_shap = pd.DataFrame(
    imputer.fit_transform(train_full[use_features]), columns=use_features
)
X_test_shap = pd.DataFrame(
    imputer.transform(test[use_features]), columns=use_features
)

X_train_res, y_train_res, pos_weight = apply_resampling(
    X_train_shap, y_train_full, METHOD, SMOTE_RATIO
)
print(f"리샘플링 적용({METHOD}, SMOTE_Ratio={SMOTE_RATIO}): "
      f"{len(X_train_shap)}행 -> {len(X_train_res)}행, pos_weight={pos_weight:.4f}")

model = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=RANDOM_STATE, verbosity=0,
    scale_pos_weight=pos_weight
)
model.fit(X_train_res, y_train_res)


# ============================================================
# 5. SHAP 계산 (메모리 내에서만 사용, CSV 저장 안 함)
# ============================================================
print("SHAP 계산 중...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_shap)

shap_importance = pd.DataFrame({
    "Feature"       : use_features,
    "mean_abs_SHAP" : np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
shap_importance["Rank"] = shap_importance.index + 1

# ── PNG 1: SHAP Bar Plot ──────────────────────────────────
plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
shap.summary_plot(shap_values, X_test_shap, plot_type="bar", show=False, max_display=20)
plt.title(f"SHAP Global Feature Importance (Bar)\n{FEATURE_SET} ({METHOD})", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_sub, f"shap_bar_{FEATURE_SET}.png"), dpi=150, bbox_inches="tight")
plt.close()

# ── PNG 2: SHAP Beeswarm Plot ─────────────────────────────
plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
shap.summary_plot(shap_values, X_test_shap, plot_type="dot", show=False, max_display=20)
plt.title(f"SHAP Beeswarm Plot\n{FEATURE_SET} ({METHOD})", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_sub, f"shap_beeswarm_{FEATURE_SET}.png"), dpi=150, bbox_inches="tight")
plt.close()


# ============================================================
# 6. Permutation Importance (메모리 내에서만 사용)
# ============================================================
print("Permutation Importance 계산 중...")
perm_result = permutation_importance(
    model, X_test_shap, y_test,
    n_repeats=30, random_state=RANDOM_STATE,
    scoring="average_precision", n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature"   : use_features,
    "Perm_Mean" : perm_result.importances_mean,
    "Perm_Std"  : perm_result.importances_std,
}).sort_values("Perm_Mean", ascending=False).reset_index(drop=True)
perm_df["Rank"] = perm_df.index + 1

# ── PNG 3: Permutation Bar Plot ───────────────────────────
top_n    = min(20, len(use_features))
perm_top = perm_df.head(top_n).sort_values("Perm_Mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
ax.barh(
    perm_top["Feature"], perm_top["Perm_Mean"],
    xerr=perm_top["Perm_Std"],
    color="#2F6EBA", alpha=0.8,
    error_kw=dict(ecolor="#555555", capsize=3)
)
ax.axvline(0, color="red", linestyle="--", linewidth=1.0)
ax.set_xlabel("Mean decrease in PR_AUC", fontsize=10)
ax.set_title(f"Permutation Importance (Top {top_n})\n{FEATURE_SET} ({METHOD})", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(save_sub, f"permutation_bar_{FEATURE_SET}.png"), dpi=150, bbox_inches="tight")
plt.close()


# ============================================================
# 7. feature_analysis_{FEATURE_SET}.csv 생성 (유일한 CSV 저장)
# ============================================================
FEATURE_CATEGORY = {
    # 안정성 (Solvency)
    "자본잠식률"                        : "안정성 (Solvency)",
    "비유동장기적합률_ratio"             : "안정성 (Solvency)",
    "차입금의존도_diff_industry"         : "안정성 (Solvency)",
    "부채비율"                          : "안정성 (Solvency)",
    "자기자본비율_diff_industry"         : "안정성 (Solvency)",
    "부채비율변화"                       : "안정성 (Solvency)",
    "장기부채의존도"                     : "안정성 (Solvency)",
    "유동비율변화_diff"                  : "안정성 (Solvency)",
    "유동비율_ratio"                     : "안정성 (Solvency)",
    "장기부채비율"                       : "안정성 (Solvency)",
    "유보율_diff"                        : "안정성 (Solvency)",
    "순운전자본비율_ratio"               : "안정성 (Solvency)",
    "순운전자본대총자본_ratio_industry"  : "안정성 (Solvency)",

    # 수익성 (Profitability)
    "총자본영업이익률_diff"              : "수익성 (Profitability)",
    "금융비용부담률"                     : "수익성 (Profitability)",
    "ROA변화"                           : "수익성 (Profitability)",
    "매출액순이익률_diff_industry"       : "수익성 (Profitability)",
    "ROIC_diff"                         : "수익성 (Profitability)",
    "ROA_ratio"                         : "수익성 (Profitability)",
    "순이익률_ratio"                     : "수익성 (Profitability)",
    "현금ROA"                           : "수익성 (Profitability)",
    "매출총이익률_diff"                  : "수익성 (Profitability)",
    "매출원가율"                         : "수익성 (Profitability)",
    "ROE_diff"                          : "수익성 (Profitability)",
    "현금ROE_ratio"                     : "수익성 (Profitability)",

    # 성장성 (Growth)
    "매출액증가율"                       : "성장성 (Growth)",
    "순이익증가율_diff"                  : "성장성 (Growth)",
    "유형자산증가율_ratio_industry"      : "성장성 (Growth)",
    "총자산증가율_diff_industry"         : "성장성 (Growth)",
    "자기자본증가율"                     : "성장성 (Growth)",

    # 활동성 (Activity)
    "비유동자산회전율_ratio"             : "활동성 (Activity)",
    "유형자산회전율_diff"                : "활동성 (Activity)",
    "매출채권회전율_ratio_industry"      : "활동성 (Activity)",
    "순운전자본회전율_diff"              : "활동성 (Activity)",
    "유동자산회전율"                     : "활동성 (Activity)",
    "총자산회전율_ratio"                 : "활동성 (Activity)",
    "재고자산보유기간_ratio"             : "활동성 (Activity)",
    "매입채무지급기간_diff"              : "활동성 (Activity)",

    # 현금흐름 (Cash Flow)
    "영업CF_유동부채_diff"               : "현금흐름 (Cash Flow)",
    "영업CF_총부채_diff"                 : "현금흐름 (Cash Flow)",
    "FCF_총자산_ratio"                   : "현금흐름 (Cash Flow)",
    "영업현금흐름비율"                   : "현금흐름 (Cash Flow)",
    "감가상각비율"                       : "현금흐름 (Cash Flow)",

    # 기타
    "업력"                              : "기타 (Other)",
    "유형자산비율"                       : "기타 (Other)",
}

CATEGORY_ORDER = {
    "안정성 (Solvency)"      : 1,
    "수익성 (Profitability)" : 2,
    "성장성 (Growth)"        : 3,
    "활동성 (Activity)"      : 4,
    "현금흐름 (Cash Flow)"   : 5,
    "기타 (Other)"           : 6,
    "미분류"                 : 7,
}

n_features = len(shap_importance)

df = shap_importance[["Rank", "Feature", "mean_abs_SHAP"]].rename(
    columns={"Rank": "SHAP_Rank"}
).merge(
    perm_df[["Feature", "Perm_Mean", "Perm_Std", "Rank"]].rename(
        columns={"Rank": "Perm_Rank"}
    ),
    on="Feature", how="inner"
)

df["SHAP_Rank_Norm"] = (df["SHAP_Rank"] - 1) / (n_features - 1)
df["Perm_Rank_Norm"] = (df["Perm_Rank"] - 1) / (n_features - 1)
df["Rank_Diff"] = (df["SHAP_Rank"] - df["Perm_Rank"]).abs()
df["Combined_Score"] = (df["SHAP_Rank_Norm"] + df["Perm_Rank_Norm"]) / 2
df["Combined_Rank"]  = df["Combined_Score"].rank(method="min").astype(int)

top_n_cls = max(3, int(n_features * 0.3))
shap_top = set(df.nsmallest(top_n_cls, "SHAP_Rank")["Feature"])
perm_top_set = set(df.nsmallest(top_n_cls, "Perm_Rank")["Feature"])

def classify(r):
    in_shap = r["Feature"] in shap_top
    in_perm = r["Feature"] in perm_top_set
    if in_shap and in_perm:
        return "★ 핵심피처 (SHAP+Perm 모두 높음)"
    elif in_shap and not in_perm:
        return "△ 대체가능 (SHAP 높음, Perm 낮음)"
    elif not in_shap and in_perm:
        return "▲ 상호작용 (Perm 높음, SHAP 낮음)"
    else:
        return "- 일반피처"

df["Feature_Type"] = df.apply(classify, axis=1)

def rank_diff_level(diff):
    if diff <= 3:
        return "일치"
    elif diff <= 8:
        return "소폭 불일치"
    else:
        return "대폭 불일치"

df["Consistency"] = df["Rank_Diff"].apply(rank_diff_level)

df["Category"]       = df["Feature"].map(FEATURE_CATEGORY).fillna("미분류")
df["Category_Order"] = df["Category"].map(CATEGORY_ORDER)

df = df[[
    "Combined_Rank", "Feature", "Category", "Feature_Type",
    "SHAP_Rank", "mean_abs_SHAP",
    "Perm_Rank", "Perm_Mean", "Perm_Std",
    "Rank_Diff", "Consistency",
    "Combined_Score", "Category_Order",
]].sort_values("Combined_Rank").reset_index(drop=True)

out_path = os.path.join(save_sub, f"feature_analysis_{FEATURE_SET}.csv")
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"\n저장 완료:")
print(f"  - {out_path}")
print(f"  - {os.path.join(save_sub, f'shap_bar_{FEATURE_SET}.png')}")
print(f"  - {os.path.join(save_sub, f'shap_beeswarm_{FEATURE_SET}.png')}")
print(f"  - {os.path.join(save_sub, f'permutation_bar_{FEATURE_SET}.png')}")

FeatureSet: top65_dedup52 | Method: ClassWeight | SMOTE_Ratio: -
피처 수: 52개
리샘플링 적용(ClassWeight, SMOTE_Ratio=-): 28111행 -> 28111행, pos_weight=25.6455
SHAP 계산 중...
Permutation Importance 계산 중...

저장 완료:
  - 16번. SHAP\전역\top65_dedup52\feature_analysis_top65_dedup52.csv
  - 16번. SHAP\전역\top65_dedup52\shap_bar_top65_dedup52.png
  - 16번. SHAP\전역\top65_dedup52\shap_beeswarm_top65_dedup52.png
  - 16번. SHAP\전역\top65_dedup52\permutation_bar_top65_dedup52.png
